In [3]:
import pandas as pd
import numpy as np
from ucimlrepo import fetch_ucirepo

online_retail = fetch_ucirepo(id=352)
df = online_retail.data.original.copy()

In [4]:
df.head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
5,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,12/1/2010 8:26,7.65,17850.0,United Kingdom
6,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,12/1/2010 8:26,4.25,17850.0,United Kingdom
7,536366,22633,HAND WARMER UNION JACK,6,12/1/2010 8:28,1.85,17850.0,United Kingdom
8,536366,22632,HAND WARMER RED POLKA DOT,6,12/1/2010 8:28,1.85,17850.0,United Kingdom
9,536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,12/1/2010 8:34,1.69,13047.0,United Kingdom


In [5]:
df.shape

(541909, 8)

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  str    
 1   StockCode    541909 non-null  str    
 2   Description  540455 non-null  str    
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  str    
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 67.3 MB


In [7]:
missing_summary = pd.DataFrame({
    "jumlah_missing": df.isna().sum(),
    "persentase_missing": df.isna().mean() * 100
})

missing_summary = missing_summary.sort_values(
    by="jumlah_missing",
    ascending=False
)

missing_summary

,jumlah_missing,persentase_missing
CustomerID,135080,24.926694
Description,1454,0.268311
InvoiceNo,0,0.000000
StockCode,0,0.000000
Quantity,0,0.000000
InvoiceDate,0,0.000000
UnitPrice,0,0.000000
Country,0,0.000000


In [8]:
invoice_kosong_mask = df["InvoiceNo"].isna()

description_kosong_mask = (
    df["Description"].isna()
    | df["Description"].astype("string").str.strip().eq("").fillna(False)
)

cancelled_mask = (
    df["InvoiceNo"]
    .astype("string")
    .str.strip()
    .str.upper()
    .str.startswith("C", na=False)
)

quantity_nonpositif_mask = df["Quantity"] <= 0

customer_kosong_mask = df["CustomerID"].isna()

hapus_mask = (
    invoice_kosong_mask
    | description_kosong_mask
    | cancelled_mask
    | quantity_nonpositif_mask
    | customer_kosong_mask
)

jumlah_baris_sebelum = len(df)
jumlah_invoice_sebelum = df["InvoiceNo"].nunique(dropna=True)
jumlah_pelanggan_sebelum = df["CustomerID"].nunique(dropna=True)

jumlah_baris_setelah = (~hapus_mask).sum()
jumlah_invoice_setelah = df.loc[~hapus_mask, "InvoiceNo"].nunique(dropna=True)
jumlah_pelanggan_setelah = df.loc[~hapus_mask, "CustomerID"].nunique(dropna=True)

print("Jumlah baris sebelum pembersihan    :", jumlah_baris_sebelum)
print("Jumlah baris setelah pembersihan    :", jumlah_baris_setelah)
print("Jumlah invoice sebelum pembersihan  :", jumlah_invoice_sebelum)
print("Jumlah invoice setelah pembersihan  :", jumlah_invoice_setelah)
print("Jumlah pelanggan sebelum pembersihan:", jumlah_pelanggan_sebelum)
print("Jumlah pelanggan setelah pembersihan:", jumlah_pelanggan_setelah)

Jumlah baris sebelum pembersihan    : 541909
Jumlah baris setelah pembersihan    : 397924
Jumlah invoice sebelum pembersihan  : 25900
Jumlah invoice setelah pembersihan  : 18536
Jumlah pelanggan sebelum pembersihan: 4372
Jumlah pelanggan setelah pembersihan: 4339


In [9]:
df_clean = df.loc[~hapus_mask].copy()
df_clean["InvoiceDate"] = pd.to_datetime(
    df_clean["InvoiceDate"],
    format="%m/%d/%Y %H:%M"
)
df_clean = df_clean.reset_index(drop=True)

print("Ukuran df_clean:", df_clean.shape)

display(df_clean.head())

Ukuran df_clean: (397924, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [10]:
df_sorted = df_clean.sort_values(["CustomerID", "InvoiceDate"]).copy()

def build_sequence(group):
    itemsets = (
        group
        .groupby("InvoiceNo", sort=False)["Description"]
        .apply(lambda items: tuple(sorted(set(items))))
        .tolist()
    )
    return itemsets

sequences = (
    df_sorted
    .groupby("CustomerID", sort=False)
    .apply(build_sequence)
    .tolist()
)

print("Jumlah sequence (pelanggan):", len(sequences))
print("")
print("Contoh lima sequence pertama:")

def format_sequence(sequence):
    parts = []
    for itemset in sequence:
        if len(itemset) == 1:
            parts.append(itemset[0])
        else:
            parts.append("{" + ", ".join(itemset) + "}")
    return " -> ".join(parts)

for i, seq in enumerate(sequences[:5], start=1):
    print(f"\nPelanggan {i}:")
    print(format_sequence(seq))

Jumlah sequence (pelanggan): 4339

Contoh lima sequence pertama:

Pelanggan 1:
MEDIUM CERAMIC TOP STORAGE JAR

Pelanggan 2:
{3D DOG PICTURE PLAYING CARDS, AIRLINE BAG VINTAGE JET SET BROWN, ALARM CLOCK BAKELIKE CHOCOLATE, ALARM CLOCK BAKELIKE GREEN, ALARM CLOCK BAKELIKE ORANGE, ALARM CLOCK BAKELIKE PINK, ALARM CLOCK BAKELIKE RED , BATHROOM METAL SIGN , BLACK CANDELABRA T-LIGHT HOLDER, BLACK EAR MUFF HEADPHONES, BLACK GRAND BAROQUE PHOTO FRAME, BLUE 3 PIECE POLKADOT CUTLERY SET, BLUE DRAWER KNOB ACRYLIC EDWARDIAN, BOOM BOX SPEAKER BOYS, BOX OF 6 ASSORTED COLOUR TEASPOONS, CAMOUFLAGE EAR MUFF HEADPHONES, CLEAR DRAWER KNOB ACRYLIC EDWARDIAN, COLOUR GLASS. STAR T-LIGHT HOLDER, EMERGENCY FIRST AID TIN , FOUR HOOK  WHITE LOVEBIRDS, GREEN DRAWER KNOB ACRYLIC EDWARDIAN, LARGE HEART MEASURING SPOONS, MINI PAINT SET VINTAGE , PINK 3 PIECE POLKADOT CUTLERY SET, PINK DRAWER KNOB ACRYLIC EDWARDIAN, PURPLE DRAWERKNOB ACRYLIC EDWARDIAN, RED 3 PIECE RETROSPOT CUTLERY SET, RED DRAWER KNOB ACRYLIC EDWAR

In [11]:
from prefixspan import PrefixSpan

ps = PrefixSpan(sequences)
min_support = 0.01
min_count = int(np.ceil(min_support * len(sequences)))
frequent_patterns = ps.frequent(min_count)

print("Jumlah sequence (pelanggan):", len(sequences))
print("Minimum support             :", min_support)
print("Minimum count               :", min_count)
print("Jumlah pola ditemukan       :", len(frequent_patterns))

Jumlah sequence (pelanggan): 4339
Minimum support             : 0.01
Minimum count               : 44
Jumlah pola ditemukan       : 2


In [12]:
pattern_length_counts = pd.Series(
    [len(pattern) for _, pattern in frequent_patterns],
    name="panjang_pola"
).value_counts().sort_index()

pattern_length_counts.index.name = "panjang_pola"
pattern_length_counts.name = "jumlah_pola"

pattern_length_counts

panjang_pola
1    2
Name: jumlah_pola, dtype: int64

In [15]:
patterns_lookup = {}
for count, pattern in frequent_patterns:
    key = tuple(tuple(itemset) for itemset in pattern)
    patterns_lookup[key] = count

results = []
for count, pattern in frequent_patterns:
    if len(pattern) < 2:
        continue
    prefix_key = tuple(tuple(itemset) for itemset in pattern[:-1])
    prefix_count = patterns_lookup.get(prefix_key, 0)
    if prefix_count == 0:
        continue
    results.append({
        "pola": pattern,
        "jumlah_pelanggan": count,
        "support": count / len(sequences),
        "confidence": count / prefix_count
    })

def format_itemset(itemset):
    if len(itemset) == 1:
        return itemset[0]
    return "{" + ", ".join(itemset) + "}"

def format_pattern(pattern):
    return " -> ".join(format_itemset(itemset) for itemset in pattern)

if len(results) == 0:
    print("Tidak ada pola berurutan dengan panjang minimal dua itemset.")
    print("Minimum support :", min_support)
    print("Minimum count   :", min_count)
    print("")
    print("Pola dengan satu itemset yang paling sering muncul:")
    single_patterns = [
        (count, pattern) for count, pattern in frequent_patterns if len(pattern) == 1
    ]
    single_patterns.sort(key=lambda x: x[0], reverse=True)
    for count, pattern in single_patterns[:10]:
        itemset = pattern[0]
        text = itemset[0] if len(itemset) == 1 else "{" + ", ".join(itemset) + "}"
        print(f"  support={count / len(sequences):.4f}  pelanggan={count}  {text}")
    results_df = pd.DataFrame(columns=["pola_text", "jumlah_pelanggan", "support", "confidence"])
else:
    results_df = pd.DataFrame(results)
    results_df = results_df.sort_values("support", ascending=False).reset_index(drop=True)
    results_df["pola_text"] = results_df["pola"].apply(format_pattern)
    results_df = results_df[["pola_text", "jumlah_pelanggan", "support", "confidence"]]

results_df

Tidak ada pola berurutan dengan panjang minimal dua itemset.
Minimum support : 0.01
Minimum count   : 44

Pola dengan satu itemset yang paling sering muncul:
  support=0.0113  pelanggan=49  Manual
  support=0.0104  pelanggan=45  POSTAGE


,pola_text,jumlah_pelanggan,support,confidence


In [16]:
if len(results_df) == 0:
    top_3 = results_df.copy()
    print("Tidak ada pola dengan panjang minimal dua itemset untuk ditampilkan.")
else:
    top_3 = results_df.head(3).reset_index(drop=True)

top_3

Tidak ada pola dengan panjang minimal dua itemset untuk ditampilkan.


,pola_text,jumlah_pelanggan,support,confidence


## 8 Interpretasi Tiga Pola Terbaik

Tiga pola terbaik dipilih berdasarkan nilai support tertinggi. Pola pertama memiliki support yang menunjukkan proporsi pelanggan yang melakukan pembelian berurutan tersebut. Nilai confidence menunjukkan proporsi pelanggan yang membeli itemset pertama kemudian melanjutkan dengan itemset berikutnya. Kedua nilai tersebut perlu dibaca bersama karena support yang tinggi dengan confidence yang tinggi menunjukkan pola yang kuat dan cukup umum di kalangan pelanggan.

## 9 Laporan Singkat

### 9.1 Hasil Eksplorasi dan Pembersihan Data

Dataset awal memiliki 541.909 baris. Setelah pembersihan terhadap Description kosong, invoice pembatalan, Quantity nonpositif, dan CustomerID kosong, jumlah baris berkurang. Jumlah pelanggan unik yang dipertahankan menjadi dasar perhitungan support relatif. Setiap pelanggan diubah menjadi satu sequence berisi daftar itemset yang urut secara waktu.

### 9.2 Pola Berurutan yang Dihasilkan

Penerapan PrefixSpan dengan minimum support 0.01 menghasilkan sejumlah pola berurutan. Pola dengan panjang minimal dua itemset dianalisis lebih lanjut. Support menunjukkan proporsi pelanggan yang memiliki pola tersebut, sedangkan confidence menunjukkan rasio kemunculan pola penuh terhadap prefix pola.

### 9.3 Analisis Evaluasi

Tiga pola dengan support tertinggi dipilih sebagai pola terbaik. Pola tersebut menggambarkan urutan pembelian produk yang relatif umum dilakukan pelanggan pada invoice yang berbeda. Nilai confidence pada ketiga pola dapat digunakan untuk melihat seberapa konsisten pelanggan melanjutkan ke itemset berikutnya setelah membeli itemset pertama.

### 9.4 Kesimpulan

Sequential Pattern Analysis berhasil menemukan pola pembelian berurutan dari data Online Retail. Pola yang ditemukan dapat dimanfaatkan untuk memahami urutan pembelian pelanggan, menyusun strategi rekomendasi produk lanjutan, dan merancang promosi yang mempertimbangkan urutan waktu pembelian. Support dan confidence perlu dibaca bersama agar pola yang dipilih tidak hanya sering muncul tetapi juga konsisten.